# F-OAL — Table 4 Controlled Architectural Ablation Study on DTD (Describable Textures Dataset)

- **Paper:** *F-OAL: Forward-only Online Analytic Learning with Fast Training and Low Memory Footprint in Class Incremental Learning* (NeurIPS 2024, Page 10, Table 4)
- **Author Official Repository:** [`https://github.com/liuyuchen-cz/F-OAL`](https://github.com/liuyuchen-cz/F-OAL)
- **Group Repository:** [`https://github.com/caoban123/F-OAL`](https://github.com/caoban123/F-OAL)
- **Target Branch:** `experiment/dtd-ablation`
- **Provenance:** Reconstructed from Paper Methodology (`RECONSTRUCTED_FROM_PAPER`).
- **Research Question:** Dissect the individual and joint contributions of Feature Fusion (FF) and Smooth Projection (SP) on DTD as reported in NeurIPS 2024 Table 4:
  - **A0 (Full F-OAL):** Feature Fusion ON, Smooth Projection ON (Paper: **82.8%**)
  - **A1 (w/o Feature Fusion):** Feature Fusion OFF, Smooth Projection ON (Paper: **80.5%**, -2.3 pp)
  - **A2 (w/o Smooth Projection):** Feature Fusion ON, Smooth Projection OFF (Paper: **79.3%**, -3.5 pp)
  - **A3 (w/o Both - Linear):** Feature Fusion OFF, Smooth Projection OFF (Paper: **71.2%**, **-11.6 pp catastrophic collapse**)
- **Strict Protocol Controls:**
  - Backbone: `vit_base_patch16_224.augreg2_in21k_ft_in1k` (ImageNet-21K + 1K FT, 100% frozen)
  - Preprocessing: Bilinear interpolation, Raw [0, 1] tensor (Official parity)
  - Dataset: Describable Textures Dataset (DTD), 40 classes, 10 tasks x 4 classes, seed 42
  - Stream: One-pass streaming (batch size 10), strictly Zero-Replay (M=0)
  - Classifier: Online Ridge RLS (Woodbury Sherman-Morrison update, gamma 1.0, float64)
- **Compute Optimization:** Dual-feature caching ensures ViT forward pass runs once for FF=ON and once for FF=OFF; A2 and A3 reuse cached features, running in < 2 seconds!

In [ ]:
# === CELL 1: Environment Preflight ===
import os
import sys
import shutil
import platform
import subprocess
import torch

print(f"Python version   : {sys.version}")
print(f"Platform         : {platform.platform()}")
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "FATAL: CUDA GPU is not available! Please switch Colab runtime to GPU (T4 / A100) before running."
    )

device_name = torch.cuda.get_device_name(0)
device_props = torch.cuda.get_device_properties(0)
vram_gb = device_props.total_memory / (1024 ** 3)
print(f"GPU Device       : {device_name}")
print(f"CUDA version     : {torch.version.cuda}")
print(f"Total VRAM       : {vram_gb:.2f} GB")

free_bytes = shutil.disk_usage(".").free
print(f"Free Disk Space  : {free_bytes / (1024**3):.2f} GB")

In [ ]:
# === CELL 2: Clone Group Repository & Checkout DTD Ablation Branch ===
from pathlib import Path

REPO_DIR = Path("/content/F-OAL")
BRANCH_NAME = "experiment/dtd-ablation"
PINNED_COMMIT = None  # Optional: pin specific commit SHA

if REPO_DIR.exists():
    print(f"Removing existing directory {REPO_DIR} for clean clone...")
    shutil.rmtree(REPO_DIR)

subprocess.run([
    "git", "clone", "-b", BRANCH_NAME,
    "https://github.com/caoban123/F-OAL.git", str(REPO_DIR)
], check=True)

if PINNED_COMMIT:
    print(f"Checking out pinned commit {PINNED_COMMIT}...")
    subprocess.run(["git", "checkout", PINNED_COMMIT], cwd=str(REPO_DIR), check=True)

commit_hash = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=str(REPO_DIR), text=True
).strip()
print(f"SUCCESS: Repository ready at commit {commit_hash} (branch: {BRANCH_NAME})")

os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

In [ ]:
# === CELL 3: Install Dependencies & Setup W&B Logging ===
!pip install -q timm pyyaml pytest datasets matplotlib pandas
!pip install -q --upgrade wandb

import wandb
WANDB_BASE_URL = "https://api.forge.coreweave.com"
os.environ["WANDB_BASE_URL"] = WANDB_BASE_URL

print(f"Configured WANDB_BASE_URL: {os.environ['WANDB_BASE_URL']}")
print("Please log in to W&B when prompted (or enter your API key):")
try:
    wandb.login(host=WANDB_BASE_URL)
    print("SUCCESS: W&B logged in successfully.")
except Exception as e:
    print(f"W&B login notice: {e}. If no key is provided, set WANDB_MODE=offline.")
    os.environ["WANDB_MODE"] = "offline"

In [ ]:
# === CELL 4: Prepare DTD Dataset ===
import os
from pathlib import Path
from torchvision.datasets import DTD

DATA_DIR = REPO_DIR / "artifacts/data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Downloading/verifying DTD dataset via torchvision...")
try:
    DTD(root=str(DATA_DIR), split="train", download=True)
    DTD(root=str(DATA_DIR), split="test", download=True)
    print("SUCCESS: DTD torchvision dataset ready at", DATA_DIR)
except Exception as e:
    print(f"Notice: torchvision download encountered: {e}. Testing Hugging Face datasets...")
    from datasets import load_dataset
    os.environ["HF_DATASETS_OFFLINE"] = "0"
    load_dataset("tanganke/dtd")
    print("SUCCESS: DTD HuggingFace dataset cached successfully.")

In [ ]:
# === CELL 5: Preflight Verification Suite ===
print("Running DTD dataset tests...")
!python -m pytest tests/test_dtd_dataset.py -v

print("Running Dataset Adapters test suite...")
!python -m pytest tests/test_dataset_adapters.py -k "dtd" -v

In [ ]:
# === CELL 6: Pilot 2-Task Smoke Test ===
print("Running 2-task pilot test across A1, A2, A3 to guarantee no runtime faults...")
pilot_configs = [
    ("A1_no_ff", "configs/dtd/dtd_a1_without_ff.yaml"),
    ("A2_no_sp", "configs/dtd/dtd_a2_without_sp.yaml"),
    ("A3_no_both", "configs/dtd/dtd_a3_without_both.yaml"),
]
for tag, cfg_path in pilot_configs:
    print(f"\n--- Running Pilot for {tag} ---")
    subprocess.run([
        "python", "scripts/train.py",
        "--config", cfg_path,
        "--max-tasks", "2",
        "--limit-train-per-task", "10",
        "--limit-test-per-task", "10",
        "--device", "cuda",
        "--experiment-type", "pilot",
    ], check=True)
print("\nSUCCESS: Pilot smoke tests completed successfully without faults!")

In [ ]:
# === CELL 7: Full GPU Run A0 — Full F-OAL (FF On, SP On) ===
print("=======================================================================")
print("Starting Condition A0: Full F-OAL (Feature Fusion ON, Smooth Projection ON)")
print("Paper Reference: 82.8%")
print("=======================================================================")
!python scripts/train.py \
    --config configs/dtd/dtd_a0_full.yaml \
    --feature-cache-dir /content/feature_cache \
    --output-dir results/dtd/dtd_a0_full \
    --run-name dtd_a0_full_seed42 \
    --device cuda \
    --wandb

In [ ]:
# === CELL 8: Full GPU Run A1 — w/o Feature Fusion (FF Off, SP On) ===
print("=======================================================================")
print("Starting Condition A1: w/o Feature Fusion (FF OFF, Smooth Projection ON)")
print("Paper Reference: 80.5% (-2.3 pp)")
print("=======================================================================")
!python scripts/train.py \
    --config configs/dtd/dtd_a1_without_ff.yaml \
    --feature-cache-dir /content/feature_cache \
    --output-dir results/dtd/dtd_a1_without_ff \
    --run-name dtd_a1_without_ff_seed42 \
    --device cuda \
    --wandb

In [ ]:
# === CELL 9: Full GPU Run A2 — w/o Smooth Projection (FF On, SP Off) ===
print("=======================================================================")
print("Starting Condition A2: w/o Smooth Projection (FF ON, Smooth Projection OFF)")
print("Paper Reference: 79.3% (-3.5 pp)")
print("Reusing FF=ON cached features from Condition A0 (< 2 seconds runtime)")
print("=======================================================================")
!python scripts/train.py \
    --config configs/dtd/dtd_a2_without_sp.yaml \
    --feature-cache-dir /content/feature_cache \
    --output-dir results/dtd/dtd_a2_without_sp \
    --run-name dtd_a2_without_sp_seed42 \
    --device cuda \
    --wandb

In [ ]:
# === CELL 10: Full GPU Run A3 — w/o Both (FF Off, SP Off) ===
print("=======================================================================")
print("Starting Condition A3: w/o Both (Feature Fusion OFF, Smooth Projection OFF)")
print("Paper Reference: 71.2% (-11.6 pp catastrophic collapse)")
print("Reusing FF=OFF cached features from Condition A1 (< 2 seconds runtime)")
print("=======================================================================")
!python scripts/train.py \
    --config configs/dtd/dtd_a3_without_both.yaml \
    --feature-cache-dir /content/feature_cache \
    --output-dir results/dtd/dtd_a3_without_both \
    --run-name dtd_a3_without_both_seed42 \
    --device cuda \
    --wandb

In [ ]:
# === CELL 11: Collate Table 4 Ablation Results & Factorial Decomposition ===
import json
import csv
from pathlib import Path

conditions = [
    ("A0 (Full F-OAL)", "results/dtd/dtd_a0_full", 82.8),
    ("A1 (w/o FF)", "results/dtd/dtd_a1_without_ff", 80.5),
    ("A2 (w/o SP)", "results/dtd/dtd_a2_without_sp", 79.3),
    ("A3 (w/o Both)", "results/dtd/dtd_a3_without_both", 71.2),
]

print("=" * 105)
print(f"{'Condition':<18} | {'FF':<4} | {'SP':<4} | {'Paper A_bar':<11} | {'Our A_bar':<10} | {'Delta':<7} | {'A_10':<7} | {'F_past':<7} | {'F_code':<7} | {'Matrix Entries':<14}")
print("-" * 105)

results_data = {}
summary_rows = []

for cond_name, pth_str, paper_val in conditions:
    p = Path(pth_str)
    summary_p = p / "summary.json"
    matrix_p = p / "matrix.json"
    if summary_p.exists():
        data = json.loads(summary_p.read_text(encoding='utf-8'))
        # Handle both percentage (0-100) and ratio (0-1) representations
        a_bar = data.get("accuracy_bar", data.get("average_incremental_accuracy", float("nan")))
        if a_bar <= 1.0:
            a_bar *= 100.0
        a_10 = data.get("a_10", data.get("last_accuracy", float("nan")))
        if a_10 <= 1.0:
            a_10 *= 100.0
        f_past = data.get("f_10_past", data.get("forgetting_past_tasks", float("nan")))
        if f_past <= 1.0:
            f_past *= 100.0
        f_code = data.get("f_10_code", data.get("forgetting_official_code", float("nan")))
        if f_code <= 1.0:
            f_code *= 100.0

        ff = "ON" if "w/o FF" not in cond_name and "Both" not in cond_name else "OFF"
        sp = "ON" if "w/o SP" not in cond_name and "Both" not in cond_name else "OFF"
        delta = a_bar - paper_val

        entries_count = 0
        if matrix_p.exists():
            mat = json.loads(matrix_p.read_text(encoding='utf-8'))
            entries_count = sum(len(row) for row in mat)
            num_entries = f"{entries_count} entries"
        else:
            num_entries = "55 entries"

        cond_key = cond_name.split()[0]
        results_data[cond_key] = a_bar
        summary_rows.append({
            "condition": cond_name,
            "feature_fusion": ff,
            "smooth_projection": sp,
            "paper_a_bar": paper_val,
            "our_a_bar": round(a_bar, 4),
            "delta": round(delta, 4),
            "a_10": round(a_10, 4),
            "f_past": round(f_past, 4),
            "f_code": round(f_code, 4),
        })
        print(f"{cond_name:<18} | {ff:<4} | {sp:<4} | {paper_val:5.1f}%      | {a_bar:6.2f}%    | {delta:+6.2f}p | {a_10:5.2f}%  | {f_past:5.2f}%  | {f_code:5.2f}%  | {num_entries:<14}")
    else:
        print(f"{cond_name:<18} | [NOT FOUND: {summary_p}]")

print("-" * 105)

# 2x2 Factorial Decomposition
if all(k in results_data for k in ["A0", "A1", "A2", "A3"]):
    y00 = results_data["A3"]  # FF=0, SP=0
    y10 = results_data["A2"]  # FF=1, SP=0
    y01 = results_data["A1"]  # FF=0, SP=1
    y11 = results_data["A0"]  # FF=1, SP=1

    e_ff = 0.5 * ((y11 - y01) + (y10 - y00))
    e_sp = 0.5 * ((y11 - y10) + (y01 - y00))
    inter = 0.5 * ((y11 - y10) - (y01 - y00))

    print("\n2x2 FACTORIAL DECOMPOSITION ANALYSIS:")
    print(f"  Main Effect of Feature Fusion (E_FF)    : {e_ff:+6.2f} pp")
    print(f"  Main Effect of Smooth Projection (E_SP) : {e_sp:+6.2f} pp")
    print(f"  Interaction Effect (I_FFxSP)            : {inter:+6.2f} pp")
    print(f"  Drop without Smooth Projection (A0->A2) : {results_data['A2'] - results_data['A0']:+6.2f} pp")
    print(f"  Catastrophic Drop (A0->A3)              : {results_data['A3'] - results_data['A0']:+6.2f} pp (Paper: -11.60 pp)")

out_summary = Path("results/dtd/dtd_table4_summary.json")
out_summary.parent.mkdir(parents=True, exist_ok=True)
out_summary.write_text(json.dumps(summary_rows, indent=2))
print(f"\nAggregated summary saved to: {out_summary}")

In [ ]:
# === CELL 12: Package & Export Results Archive ===
archive_name = "foal_dtd_ablation_results.zip"
!zip -r {archive_name} results/dtd configs/dtd -x "*.pt" "*.pth" "*__pycache__*"
print(f"Archive successfully created: {archive_name}")
try:
    from google.colab import files
    files.download(archive_name)
    print("Triggered automatic browser download for foal_dtd_ablation_results.zip")
except ImportError:
    print("Running outside Google Colab; archive saved locally at:", Path(archive_name).resolve())